# Tutorial 2: KL Divergence Bound for Attention

This notebook demonstrates the sharp cubic remainder bound for
the KL divergence of attention weights under perturbation:

$$|\text{KL}(\alpha(\tau) \| \alpha^0) - \tfrac{\tau^2}{2} \text{Var}_{\alpha^0}(a)| \leq C |\tau|^3 \Delta^3$$

where $C = 1/(18\sqrt{3})$ for forward KL.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ghosts.theory import (
    softmax, klDivergence, computeAttentionKL,
    computeVariance, computeSlopeSpread,
    klRemainderBound, verifyBound,
    KL_REMAINDER_CONSTANT,
)

## 1. Quadratic approximation vs exact KL

For small $\tau$, KL grows quadratically. The remainder is cubic.

In [ ]:
alpha0 = softmax(np.array([[1.0, 2.0, 0.5]]))
a = np.array([[0.3, -0.5, 0.8]])
var0 = computeVariance(alpha0, a)
delta = computeSlopeSpread(a)

taus = np.linspace(0, 1.5, 200)
klExact = [computeAttentionKL(alpha0, a, t) for t in taus]
klQuad = [0.5 * t**2 * var0 for t in taus]
bound = [klRemainderBound(t, delta) for t in taus]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(taus, klExact, 'k-', lw=2, label='Exact KL')
ax.plot(taus, klQuad, 'r--', lw=1.5, label=r'Quadratic $\frac{\tau^2}{2}$Var')
ax.set_xlabel(r'$\tau$')
ax.set_ylabel('KL divergence')
ax.set_title('KL vs quadratic approximation')
ax.legend()

ax = axes[1]
remainder = [abs(e - q) for e, q in zip(klExact, klQuad)]
ax.plot(taus, remainder, 'k-', lw=2, label='|Remainder|')
ax.plot(taus, bound, 'r--', lw=1.5, label=r'Bound $C|\tau|^3\Delta^3$')
ax.set_xlabel(r'$\tau$')
ax.set_ylabel('Magnitude')
ax.set_title('Remainder vs sharp bound')
ax.legend()

plt.tight_layout()
plt.show()

## 2. Bound validity across random inputs

The bound holds for all inputs. Let's verify on 100 random samples.

In [ ]:
rng = np.random.default_rng(42)
ratios = []
for _ in range(100):
    n = rng.integers(2, 8)
    z = rng.standard_normal((1, n))
    aVec = rng.standard_normal((1, n))
    tau = rng.uniform(0.01, 0.5)
    result = verifyBound(softmax(z), aVec, tau)
    ratios.append(result["ratio"])
    assert result["valid"], f"Bound violated! ratio={result['ratio']}"

print(f"All 100 tests passed.")
print(f"Tightness ratios: min={min(ratios):.4f}, max={max(ratios):.4f}, mean={np.mean(ratios):.4f}")

## 3. Tightness at the Bernoulli extremum

The bound is tightest for the Bernoulli distribution with
$p = (3 + \sqrt{3})/6$, where the third central moment is maximized.

In [ ]:
p = (3 + np.sqrt(3)) / 6
alpha0 = np.array([[p, 1 - p]])
a = np.array([[0.0, 1.0]])

taus = np.linspace(0.01, 0.3, 50)
ratios = []
for t in taus:
    r = verifyBound(alpha0, a, t)
    ratios.append(r["ratio"])

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(taus, ratios, 'k-', lw=2)
ax.axhline(1.0, color='red', ls='--', alpha=0.5, label='Bound = 1')
ax.set_xlabel(r'$\tau$')
ax.set_ylabel('Remainder / Bound')
ax.set_title(f'Tightness at Bernoulli extremum (p={p:.4f})')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Max ratio (tightness): {max(ratios):.4f}")